In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.visualization import make_lupton_rgb

# Project import
import artpop


In [ ]:
from yasone import selection as filter_utils
from yasone import analysis as ana_utils

In [ ]:
filter_utils.load_isochrones()

In [ ]:
rng = np.random.RandomState(100)


# Utilities

In [ ]:
pixel_scale = 0.254

In [ ]:
bkg_medians = np.array([4000, 9000, 12700])

In [ ]:
bkg_fluxes = bkg_medians / pixel_scale**2

In [ ]:
sky_surface_brightnesses = [ 26.5, 25, 24.5]
image_zeropoints = [29.1, 29.2, 28.7]

In [ ]:
bkg_mags = -2.5*np.log10(bkg_fluxes) + image_zeropoints

In [ ]:
image_size = 709 # gives a 2 arcsec x 2 arcsec image

In [ ]:
bkg_mags

In [ ]:
def get_default_params(objname):
    obs_props = ana_utils.get_obs_props(objname)
    return filter_utils.filter_params(
        dm = obs_props["dm"], 
        iso_fe_h = obs_props["iso_fe_h"],
        iso_log_age = np.log10(obs_props["iso_age"]),
        objname = objname, 
        mode="gri", 
        E_BV = 0,
        r23_max_sigma=None,
        r_cen = 40/60,
        iso_width=0.15
    )
    

In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift", deredden=True)


In [ ]:
params = get_default_params("yasone1")
err_g = ana_utils.fit_err(cat["G_MAG"], cat["G_MAG_ERR"])
err_r = ana_utils.fit_err(cat["R_MAG"], cat["R_MAG_ERR"])
err_i = ana_utils.fit_err(cat["I_MAG"], cat["I_MAG_ERR"])

gr_err = filter_utils.get_gr_err(cat, params)
ri_err = filter_utils.get_ri_err(cat, params)

In [ ]:
def plot_image(imgs, stretch=20):
    rgb = make_lupton_rgb(imgs[2].image, imgs[1].image, imgs[0].image, stretch=stretch)

    
    plt.imshow(rgb, origin="lower", extent=(-xi_eta_max, xi_eta_max, -xi_eta_max, xi_eta_max));



In [ ]:
def plot_cluster(pop, imgs, params):
    fig, axs = plt.subplots(1, 2, figsize=(5, 2))
    
    plt.sca(axs[1])
    plt.scatter(pop["SDSS_r_obs"] - pop["SDSS_i_obs"], pop["SDSS_r_obs"], s=3, alpha=0.3)
    # filter_utils.plot_iso_ri(params, ri_err)
    plt.ylim(26, 16)
    plt.xlim(-1, 2)
    plt.xlabel("r - i")
    plt.ylabel("r")
    axs[1].set_box_aspect(1)
    
    
    plt.sca(axs[0])
    
    plt.scatter(pop["SDSS_g_obs"] - pop["SDSS_r_obs"], pop["SDSS_g_obs"], s=3, alpha=0.3)
    # filter_utils.plot_iso_gr(params, gr_err)
    
    plt.ylim(27, 16)
    plt.xlim(-1, 3)
    
    plt.xlabel("g - r")
    plt.ylabel("g")

    axs[0].set_box_aspect(1)

    plt.tight_layout()

In [ ]:
def sample_population_from_isochrone(
    M,
    distance = 12 * u.kpc,
    **kwargs
):
    """
    Sample a stellar population from a MIST isochrone.
    
    Parameters
    ----------
    iso : astropy.table.Table
        Must contain columns:
        'initial_mass', 'g', 'r', 'i'
    nstars : int
        Number of stars to sample
    sigma_g, sigma_r, sigma_i : functions
        Functions returning magnitude uncertainty
        given a magnitude.
    """


    spop = artpop.MISTSersicSSP(
        log_age = 10.1,        # log of age in years
        feh = -2.0,           # metallicity [Fe/H]
        r_eff = 1.5 * u.pc,   # effective radius
        n = 1,              # Sersic index
        theta = 0 * u.deg,  # position angle
        ellip = 0.0,          # ellipticity
        total_mass = M ,      # number of stars
        mass_tolerance = np.maximum(0.01, 1/M),
        phot_system = 'SDSSugriz', # photometric system
        distance=distance, # distance to system
        xy_dim = image_size,         # image dimension
        pixel_scale = pixel_scale,    # pixel scale in arcsec / pixel
        random_state = rng,   # random state for reproducibility
    )


    psf = artpop.moffat_psf(fwhm=0.7*u.arcsec)
    
    obs_time = 190 * 15 * u.second
    
    imager = artpop.ArtImager(
        phot_system = 'SDSSugriz', # photometric system
        diameter = 6 * u.m, # effective aperture diameter
        read_noise = 0,       # read noise in electrons
        random_state = rng    # random state for reproducibility
    )

    obs_g = imager.observe(spop, 'SDSS_g', obs_time, sky_sb=bkg_mags[0], psf=psf,)
    obs_r = imager.observe(spop, 'SDSS_r', obs_time, sky_sb=bkg_mags[1], psf=psf,)
    obs_i = imager.observe(spop, 'SDSS_i', obs_time, sky_sb=bkg_mags[2], psf=psf,)


    source_table = spop.mags.copy()
    source_table["SDSS_r_err"] = obs_r.mag_error
    source_table["SDSS_g_err"] = obs_g.mag_error
    source_table["SDSS_i_err"] = obs_i.mag_error

    N = len(source_table)

    source_table["SDSS_r_obs"] = source_table["SDSS_r"] + np.random.normal(0, source_table["SDSS_r_err"], N)
    source_table["SDSS_g_obs"] = source_table["SDSS_g"] + np.random.normal(0, source_table["SDSS_g_err"], N)
    source_table["SDSS_i_obs"] = source_table["SDSS_i"] + np.random.normal(0, source_table["SDSS_i_err"], N)


    return source_table, spop, (obs_g, obs_r, obs_i)

# Making image plots

In [ ]:
def combine_images(imgs):
    img_g, img_r, img_i = imgs
    img = np.stack([
            img_i.image,
            img_r.image,
            img_g.image
        ], axis=2
    )

    return img
    

In [ ]:
def color_clip_image(img, vmin=0, vmax=None, norm=np.asinh, scale=1):
    img_clipped = img.copy()
    I = np.sum(img, axis=2) / 3
    if vmax is None:
        h = np.quantile(I[np.isfinite(I)], 0.99)
    else:
        h = vmax
    l = vmin

    def f(I):
        x = (norm(I/scale) - norm(l/scale)) / (norm(h/scale) - norm(l/scale))
        return x
    for i in range(3):
        img_clipped[:, :, i] *= f(I) / I

    return img_clipped

In [ ]:
smags, spop, imgs = sample_population_from_isochrone(300)
plot_image(imgs, stretch=0.5)
plt.show()

In [ ]:
img = combine_images(imgs)
clipped = color_clip_image(img,  scale=0.5, vmin=0, vmax=10)
plt.imshow(clipped)

In [ ]:
seed = 128

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(5, 2.5), dpi=300)
rng = np.random.RandomState(seed)

for j in range(3):
    M = [300, 100, 30][j]
    for i in range(1):
    
        plt.sca(axs[j])
        smags, spop, imgs = sample_population_from_isochrone(M)

        img = combine_images(imgs)
        clipped = color_clip_image(img, vmin=0, scale=0.5, vmax=10)
        plt.imshow(clipped, origin="lower", interpolation="none")
        plt.axis('off')

        if i == 0:
            plt.title(f"${M}\\,M_\\odot$")
    


plt.tight_layout()

In [ ]:
params.iso_fe_h = "m2.00"
params.dm = 15.395906230238126

In [ ]:
err_gr = filter_utils.get_gr_err(cat, params)
err_ri = filter_utils.get_ri_err(cat, params)

In [ ]:
fig, axs = plt.subplots(2,5, figsize=(5, 2.5), dpi=300)
rng = np.random.RandomState(seed)

for j in range(5):
    M = [300, 200, 100, 50, 30][j]
    for i in range(2):
    
        plt.sca(axs[i][j])
        smags, spop, imgs = sample_population_from_isochrone(M)

        img = combine_images(imgs)
        clipped = color_clip_image(img, vmin=0, scale=3, vmax=500)
        plt.imshow(clipped, origin="lower", interpolation="none")
        plt.axis('off')

        if i == 0:
            plt.title(f"${M}\\,M_\\odot$")
    


plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(7, 2.5), sharex=True, sharey=True, gridspec_kw=dict(hspace=0, wspace=0))
rng = np.random.RandomState(seed)

for j in range(3):
    M = [300, 100, 30][j]
    for i in range(1):
        plt.sca(axs[j])

        smags, spop, imgs = sample_population_from_isochrone(M)
        plt.scatter(smags["SDSS_g_obs"] - smags["SDSS_r_obs"], smags["SDSS_g_obs"], s=9, alpha=1, lw=0)
        filter_utils.plot_iso_gr(params, err_gr)

        if i == 0:
            plt.title(f"${M}\\,M_\\odot$")

plt.xlim(-1, 2)
plt.ylim(27, 16)
plt.xticks([0, 1])
fig.supxlabel("$g - r$ (mag)")
fig.supylabel("$g$ (mag)")

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(7, 2.5), sharex=True, sharey=True, gridspec_kw=dict(hspace=0, wspace=0))
rng = np.random.RandomState(seed)

for j in range(3):
    M = [300, 100, 30][j]
    for i in range(1):
        plt.sca(axs[j])

        smags, spop, imgs = sample_population_from_isochrone(M)
        plt.scatter(smags["SDSS_r_obs"] - smags["SDSS_i_obs"], smags["SDSS_r_obs"], s=9, alpha=1, lw=0)
        if i == 0:
            plt.title(f"${M}\\,M_\\odot$")
        filter_utils.plot_iso_ri(params, err_ri)


plt.xlim(-1.2, 1.8)
plt.ylim(27, 16)

fig.supxlabel("$r - i$ (mag)")
fig.supylabel("$r$ (mag)")

# Fraction of RGB stars

# RGB probabilities

In [ ]:
def count_rgb(M, N=20, distance=12*u.kpc):
    N_rgbs = []
    
    
    for i in range(N):
        spop = artpop.MISTSersicSSP(
            log_age = 10.1,        # log of age in years
            feh = -2.0,           # metallicity [Fe/H]
            r_eff = 1.5 * u.pc,   # effective radius
            n = 1,              # Sersic index
            theta = 0 * u.deg,  # position angle
            ellip = 0.0,          # ellipticity
            total_mass = M ,      # number of stars
            mass_tolerance = np.maximum(0.01, 1/M),
            phot_system = 'SDSSugriz', # photometric system
            distance=distance, # distance to system
            xy_dim = image_size,         # image dimension
            pixel_scale = pixel_scale,    # pixel scale in arcsec / pixel
            random_state = rng,   # random state for reproducibility
        )
    
        N_rgb = np.sum((spop.mags["SDSS_r"] < 18) & (spop.mags["SDSS_r"] > 16))
        N_rgbs.append(N_rgb)

    return N_rgbs

In [ ]:
for M in [300, 100, 30, 10]:
    Ns = count_rgb(M)
    plt.scatter(M * np.random.normal(1, 0.05, size=len(Ns)), Ns, color="k", s=6, lw=0, alpha=0.3)

plt.xlabel("star cluster mass")
plt.xscale("log")
plt.ylabel("number with r > 19 mag")


In [ ]:
m_rgb_min = np.min(iso_rgb["initial_mass"])
m_rgb_max = np.max(iso_rgb["initial_mass"])
print(m_rgb_min, m_rgb_max)

In [ ]:


# A = quad(imf, 0.08, 0.7917)
# B = quad(imf, 0.77811, 0.7917)
f_giant = 0.003454738930670043

In [ ]:
from scipy.stats import binom

In [ ]:
d = binom(n=1, p=f_giant)

In [ ]:
d.cdf(1)

In [ ]:
imf_n_to_m = 2

In [ ]:
M = np.arange(1000)
N = np.arange(0, 100)
M, N = np.meshgrid(M, N)
probs = [1 - binom(n=m, p=f_giant).cdf(n) for (n, m) in zip(N, M)]
plt.contour(M, N, probs, levels=[1e-6, 1e-5, 0.0001, 0.001, 0.01, 0.05, 0.5, 1], norm="log")
plt.xscale("log")
plt.yscale('log')
plt.ylim(1, 10)
plt.xlim(1, 1e3)
plt.xlabel("number cluster stars")
plt.ylabel("number giant stars")
plt.colorbar(label="probability")


In [ ]:
iso_rgb = iso[iso["phase"] == 2]
plt.scatter(iso_rgb["SDSS_g"] - iso_rgb["SDSS_r"], iso_rgb["SDSS_g"])

iso_rgb = iso[iso["phase"] == 0]
plt.scatter(iso_rgb["SDSS_g"] - iso_rgb["SDSS_r"], iso_rgb["SDSS_g"])

plt.colorbar()
plt.ylim(8, -6)
plt.xlim(0, 1.5)

In [ ]:
Ns = count_rgb(1000)

In [ ]:
from scipy.stats import poisson

In [ ]:
plt.hist(Ns, density=True, bins=0.5 + np.arange(np.max(Ns)))

x = np.arange(0, 20)
y = [binom(n=300, p=2*f_giant).pmf(xx) for xx in x]
plt.plot(x, y)

In [ ]:
binom(n=300, p=f_giant).pmf(2) 